# RTM3D/KM3D ResNet-18 edge candidate — inference screen

**Revision:** `RTM3D-KM3D-RES18-EDGE-SCREEN-2026-10-07-r1`

This is a short, inference-only screen of the official ResNet-18 checkpoint. It checks exact upstream source, safely loads the checkpoint with a strict state-dictionary match, and runs 16 images from our Chen validation split through the model's 2D and calibration-based 3D decode. It performs no training, full-split AP evaluation, Core ML conversion, or iPhone timing.

The source is pinned to commit `888c379e79d8a6d134f06a9b7d669118679e06dc`; the official model file is `model_res18_1.pth` from the RTM3D/KM3D model zoo. We bypass the upstream all-model factory and instantiate only its plain ResNet-18 route, so this smoke does not compile DLA/DCNv2 or the training/evaluation IoU extension. The upstream decoder currently hard-codes CUDA, so select a Colab GPU runtime.

A successful result means the public checkpoint can produce finite 2D boxes and camera-coordinate 3D attributes in this smoke. It does **not** prove iPhone compatibility. The next gated step is Core ML export of the neural-network heads; the calibration geometry decode must be audited separately for a native iOS implementation.

Reference: [official RTM3D/KM3D repository](https://github.com/Banconxuan/RTM3D), [official installation notes](https://github.com/Banconxuan/RTM3D/blob/master/readme/INSTALL.md), [pretrained ResNet-18 checkpoint](https://drive.google.com/file/d/14ww6mxtitO9aDszZN3ai8N7U1doehvi8/view).

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/mobile_adas3d')
SCRIPT = PROJECT_DIR / 'scripts/run_rtm3d_km3d_res18_smoke.py'
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/rtm3d_km3d_res18_r1')
RTM3D_REPO = Path('/content/RTM3D_KM3D_R1_20261007')
EXPECTED_COMMIT = '888c379e79d8a6d134f06a9b7d669118679e06dc'
RTM3D_URL = 'https://github.com/Banconxuan/RTM3D.git'
CHECKPOINT_ID = '14ww6mxtitO9aDszZN3ai8N7U1doehvi8'

if not SCRIPT.is_file():
    raise FileNotFoundError(f'{SCRIPT} is missing. Update /content/mobile_adas3d from the project main branch, then rerun this notebook from the top.')
if not (SPLIT_DIR / 'val.txt').is_file():
    raise FileNotFoundError(SPLIT_DIR / 'val.txt')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing; this upstream geometry decoder requires CUDA.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project script:', SCRIPT)
print('GPU:', torch.cuda.get_device_name(0))
print('Output:', OUTPUT_DIR)

In [ ]:
# Clone only into this fresh candidate directory; never reset/delete an existing checkout.
if RTM3D_REPO.exists():
    head = subprocess.check_output(['git', '-C', str(RTM3D_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    if head != EXPECTED_COMMIT:
        raise RuntimeError(f'Existing source has commit {head}, expected {EXPECTED_COMMIT}. Preserve it and choose a new RTM3D_REPO path.')
else:
    subprocess.run(['git', 'clone', RTM3D_URL, str(RTM3D_REPO)], check=True)
    subprocess.run(['git', '-C', str(RTM3D_REPO), 'checkout', '--detach', EXPECTED_COMMIT], check=True)

CHECKPOINT = OUTPUT_DIR / 'model_res18_1.pth'
if not CHECKPOINT.is_file():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    subprocess.run([sys.executable, '-m', 'gdown', f'https://drive.google.com/uc?id={CHECKPOINT_ID}', '-O', str(CHECKPOINT)], check=True)
if CHECKPOINT.stat().st_size < 1_000_000:
    raise RuntimeError(f'Checkpoint download looks incomplete: {CHECKPOINT} ({CHECKPOINT.stat().st_size} bytes)')

VAL_IDS = [x.strip() for x in (SPLIT_DIR / 'val.txt').read_text().splitlines() if x.strip()]
assert len(VAL_IDS) == 3769, f'Expected 3769 Chen validation IDs, got {len(VAL_IDS)}'
DATASET_CANDIDATES = [Path('/content/kitti_m67'), Path('/content/kitti_m66'), Path('/content/kitti_m65'), Path('/content/kitti_m64'), Path('/content/kitti_m62'), Path('/content/kitti_m61'), Path('/content/kitti'), Path('/content/monodetr_kitti_a2'), Path('/content/drive/MyDrive/datasets/kitti')]
DATASET_ROOT = next((root for root in DATASET_CANDIDATES if (root / 'training/image_2' / f'{VAL_IDS[0]}.png').is_file() and (root / 'training/calib' / f'{VAL_IDS[0]}.txt').is_file()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError('Could not find the first Chen-val image/calibration. Add your KITTI root to DATASET_CANDIDATES and rerun this cell.')
print('Pinned RTM3D commit:', subprocess.check_output(['git', '-C', str(RTM3D_REPO), 'rev-parse', 'HEAD'], text=True).strip())
print('Dataset:', DATASET_ROOT)
print('Checkpoint:', CHECKPOINT, 'bytes:', CHECKPOINT.stat().st_size)

In [ ]:
# Fixed 16-image, inference-only smoke: strict weights, 2D boxes, and camera-coordinate 3D fields.
REPORT = OUTPUT_DIR / 'rtm3d_km3d_res18_fixed16_smoke.json'
command = [sys.executable, str(SCRIPT), '--repo', str(RTM3D_REPO), '--checkpoint', str(CHECKPOINT), '--dataset-root', str(DATASET_ROOT), '--split-dir', str(SPLIT_DIR), '--output', str(REPORT)]
print('+', ' '.join(command), flush=True)
subprocess.run(command, check=True)
report = json.loads(REPORT.read_text())
assert report['complete'] and report['safe_weights_only_load'] and report['strict_state_dict_load']
assert report['smoke_passed'], report
print('SMOKE PASSED; no training, full-val AP, Core ML conversion, or iPhone timing was performed.')

In [ ]:
# Package only the review report; the checkpoint stays in Drive and is not duplicated into the ZIP.
BUNDLE = OUTPUT_DIR / 'rtm3d_km3d_res18_screen_results.zip'
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT, arcname=REPORT.name)
print('Review artifact:', BUNDLE)
print('Next gate: export the neural-network heads to Core ML, then validate the calibration-based 3D decode separately. Do not connect the iPhone yet.')